In [4]:
import pandas as pd
import numpy as np

from scipy.sparse import csr_matrix
from sklearn.neighbors import NearestNeighbors
from tqdm import tqdm


SEED = 256
N_EVAL_USERS = 10000


# -----------------------------
# 1. Load parquet splits
# -----------------------------

train = pd.read_parquet("./bgg_recsys/train.parquet")
val = pd.read_parquet("./bgg_recsys/val.parquet")
test = pd.read_parquet("./bgg_recsys/test.parquet")

print(train.head())
print(train.columns)


cols = ["user", "game_id", "name", "rating"]

train = train[cols].dropna().copy()
val = val[cols].dropna().copy()
test = test[cols].dropna().copy()

train = train[train["rating"] > 0]
val = val[val["rating"] > 0]
test = test[test["rating"] > 0]


# -----------------------------
# 2. Encode users/items using train only
# -----------------------------

user_to_idx = {u: i for i, u in enumerate(train["user"].unique())}
game_to_idx = {g: i for i, g in enumerate(train["game_id"].unique())}

idx_to_game = {i: g for g, i in game_to_idx.items()}

train["user_idx"] = train["user"].map(user_to_idx)
train["game_idx"] = train["game_id"].map(game_to_idx)

val["user_idx"] = val["user"].map(user_to_idx)
val["game_idx"] = val["game_id"].map(game_to_idx)

test["user_idx"] = test["user"].map(user_to_idx)
test["game_idx"] = test["game_id"].map(game_to_idx)

# Drop validation/test rows with users or games unseen in train
val = val.dropna(subset=["user_idx", "game_idx"]).copy()
test = test.dropna(subset=["user_idx", "game_idx"]).copy()

train["user_idx"] = train["user_idx"].astype(int)
train["game_idx"] = train["game_idx"].astype(int)
val["user_idx"] = val["user_idx"].astype(int)
val["game_idx"] = val["game_idx"].astype(int)
test["user_idx"] = test["user_idx"].astype(int)
test["game_idx"] = test["game_idx"].astype(int)


N_USERS = len(user_to_idx)
N_GAMES = len(game_to_idx)

print("Train:", train.shape)
print("Val:", val.shape)
print("Test:", test.shape)
print("Users:", N_USERS)
print("Games:", N_GAMES)


# -----------------------------
# 3. Build item-user matrix
# -----------------------------

item_user_matrix = csr_matrix(
    (
        train["rating"].astype(float),
        (train["game_idx"], train["user_idx"])
    ),
    shape=(N_GAMES, N_USERS)
)


# -----------------------------
# 4. Train item-based KNN
# -----------------------------

KNN_NEIGHBORS = 50

knn = NearestNeighbors(
    metric="cosine",
    algorithm="brute",
    n_neighbors=KNN_NEIGHBORS + 1,
    n_jobs=-1
)

knn.fit(item_user_matrix)


# Precompute each game's nearest neighbors
distances, indices = knn.kneighbors(item_user_matrix)

# Convert cosine distance to cosine similarity
similarities = 1 - distances


# -----------------------------
# 5. Helper data structures
# -----------------------------

global_mean = train["rating"].mean()
item_means = train.groupby("game_idx")["rating"].mean().to_dict()

# User rating history:
# {user_idx: {game_idx: rating}}
user_rating_dict = (
    train.groupby("user_idx")
    .apply(lambda x: dict(zip(x["game_idx"], x["rating"])))
    .to_dict()
)


# -----------------------------
# 6. KNN prediction function
# -----------------------------

def predict_knn_rating(user_idx, game_idx):
    """
    Predict how a user would rate a game using item-based KNN.

    Looks at games similar to the target game.
    If the user rated those similar games, use a similarity-weighted average.
    """

    user_ratings = user_rating_dict.get(user_idx, {})

    neighbor_items = indices[game_idx]
    neighbor_sims = similarities[game_idx]

    weighted_sum = 0.0
    sim_sum = 0.0

    for neighbor_item, sim in zip(neighbor_items, neighbor_sims):
        if neighbor_item == game_idx:
            continue

        if neighbor_item in user_ratings:
            rating = user_ratings[neighbor_item]
            weighted_sum += sim * rating
            sim_sum += abs(sim)

    if sim_sum > 0:
        return weighted_sum / sim_sum

    # Fallback if user has not rated similar games
    return item_means.get(game_idx, global_mean)


def knn_score_fn(user_idx, game_idx_array):
    """
    Used by ranking evaluation.

    Higher score = more recommended.
    """
    return np.array([
        predict_knn_rating(user_idx, int(game_idx))
        for game_idx in game_idx_array
    ])


# -----------------------------
# 7. RMSE evaluation
# -----------------------------

def rmse(preds, actuals):
    preds = np.asarray(preds)
    actuals = np.asarray(actuals)
    return float(np.sqrt(np.mean((preds - actuals) ** 2)))


def evaluate_rmse(eval_df, name="Validation"):
    preds = [
        predict_knn_rating(int(row.user_idx), int(row.game_idx))
        for row in tqdm(eval_df.itertuples(), total=len(eval_df), desc=f"{name} RMSE")
    ]

    actuals = eval_df["rating"].values

    return rmse(preds, actuals)


val_rmse = evaluate_rmse(val, "Validation")
test_rmse = evaluate_rmse(test, "Test")

print("Validation RMSE:", val_rmse)
print("Test RMSE:", test_rmse)


# -----------------------------
# 8. Ranking evaluation harness
# -----------------------------

def _sample_eval_subset(test_df, n=N_EVAL_USERS, seed=SEED):
    if n is None or n >= len(test_df):
        return test_df
    return test_df.sample(n=n, random_state=seed).reset_index(drop=True)


def _build_user_seen_arrays(train_df):
    user_seen = {}
    for u, grp in train_df.groupby("user_idx", sort=False)["game_idx"]:
        user_seen[int(u)] = grp.values.astype(np.int32)
    return user_seen


def evaluate_ranking(model_score_fn, test_df, train_df, n_negatives=99, k=10, seed=SEED):
    rng = np.random.RandomState(seed)
    eval_df = _sample_eval_subset(test_df)
    user_seen = _build_user_seen_arrays(train_df)

    oversample = int(n_negatives * 1.5) + 5

    recalls, ndcgs = [], []

    users = eval_df["user_idx"].values.astype(np.int64)
    positives = eval_df["game_idx"].values.astype(np.int64)

    for u, pos in tqdm(zip(users, positives), total=len(users), desc="Ranking eval"):
        seen = user_seen.get(int(u), np.array([], dtype=np.int32))

        cand = rng.randint(0, N_GAMES, size=oversample)
        mask = ~np.isin(cand, seen) & (cand != pos)
        cand = cand[mask]

        while len(cand) < n_negatives:
            extra = rng.randint(0, N_GAMES, size=oversample)
            extra = extra[~np.isin(extra, seen) & (extra != pos)]
            cand = np.concatenate([cand, extra])

        negs = cand[:n_negatives]
        candidates = np.concatenate([[pos], negs])

        scores = model_score_fn(int(u), candidates)

        rank = int((scores > scores[0]).sum())

        if rank < k:
            recalls.append(1.0)
            ndcgs.append(1.0 / np.log2(rank + 2))
        else:
            recalls.append(0.0)
            ndcgs.append(0.0)

    return {
        "recall@10": float(np.mean(recalls)),
        "ndcg@10": float(np.mean(ndcgs)),
        "n_eval": len(users)
    }


# -----------------------------
# 9. Run ranking evaluation
# -----------------------------

val_ranking = evaluate_ranking(
    model_score_fn=knn_score_fn,
    test_df=val,
    train_df=train,
    n_negatives=99,
    k=10,
    seed=SEED
)

test_ranking = evaluate_ranking(
    model_score_fn=knn_score_fn,
    test_df=test,
    train_df=train,
    n_negatives=99,
    k=10,
    seed=SEED
)

print("Validation ranking:", val_ranking)
print("Test ranking:", test_ranking)


# -----------------------------
# 10. Final results table
# -----------------------------

results = pd.DataFrame([
    {
        "Model": "Item-based KNN",
        "Val RMSE": val_rmse,
        "Test RMSE": test_rmse,
        "Val Recall@10": val_ranking["recall@10"],
        "Val NDCG@10": val_ranking["ndcg@10"],
        "Test Recall@10": test_ranking["recall@10"],
        "Test NDCG@10": test_ranking["ndcg@10"],
        "N Eval Users": test_ranking["n_eval"]
    }
])

results

      user  rating  game_id                  name  user_idx  game_idx
0  Hessu68     4.5    57422         Climate-Poker     52597      9435
1     Doel     9.0   120547           War & Peace     32710     11338
2    butze     6.0      915  Mystery of the Abbey    156861       736
3   dixijo     7.0      826             Cartagena    170065       676
4    pence     4.0   119407            Dixit Jinx    231508     11298
Index(['user', 'rating', 'game_id', 'name', 'user_idx', 'game_idx'], dtype='object')
Train: (18171881, 6)
Val: (272319, 6)
Test: (272319, 6)
Users: 272319
Games: 21802


/var/folders/x2/lntl00mj3svc1sbd717rwg300000gn/T/ipykernel_4428/313675009.py:122: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  train.groupby("user_idx")
Test RMSE: 100%|██████████| 272319/272319 [00:03<00:00, 78461.91it/s] 


Validation RMSE: 1.4444813411142297
Test RMSE: 1.4408660875702035


Ranking eval: 100%|██████████| 10000/10000 [00:07<00:00, 1264.94it/s]


Validation ranking: {'recall@10': 0.4763, 'ndcg@10': 0.24450033401794352, 'n_eval': 10000}
Test ranking: {'recall@10': 0.4764, 'ndcg@10': 0.2433912767256189, 'n_eval': 10000}


,Model,Val RMSE,Test RMSE,Val Recall@10,Val NDCG@10,Test Recall@10,Test NDCG@10,N Eval Users
0,Item-based KNN,1.444481,1.440866,0.4763,0.2445,0.4764,0.243391,10000
